In [1]:
import pandas as pd
import os
# from transformers import BertTokenizer, BertModel
from bert_score import BERTScorer
import re
import pickle
import nltk
from nltk.translate import meteor
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from textblob import TextBlob
import math

/Users/isabel/anaconda3/envs/entityRecognitionNotes/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
nltk.download('all')

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /Users/isabel/nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package alpino to
[nltk_data]    |     /Users/isabel/nltk_data...
[nltk_data]    |   Package alpino is already up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /Users/isabel/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /Users/isabel/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_eng is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /Users/isabel/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_ru is already
[nltk_data]    |       up-to-date!
[nlt

True

In [3]:
gpt3_load = './generatedDataFromGoogleDrive_gpt3/data/'
gpt4_load = './generatedDataFromGoogleDrive_gpt4/data/'
gpt3_save_bert = './metrics/gpt3_F1_BERT_scores.pkl'
gpt4_save_bert = './metrics/gpt4_F1_BERT_scores.pkl'
gpt3_save_meteor = './metrics/gpt3_meteor_scores.pkl'
gpt4_save_meteor = './metrics/gpt4_meteor_scores.pkl'
gpt3_save_sentiment = './metrics/gpt3_sentiment_scores.pkl'
gpt4_save_sentiment = './metrics/gpt4_sentiment_scores.pkl'

# Contextual Similarity - BERTScore

In [4]:
def bertScore(loadingFile, savingFile):
    # load all files from data folder (generated in Google Colab)
    met_dfs = {}
    unmet_dfs = {}
    for file in os.listdir(loadingFile):
        if '.csv' in file:
            
            if 'unmet' in file:
                unmet_dfs[file.replace('.csv', '')] = pd.read_csv(f'{loadingFile}{file}')
            else:
                met_dfs[file.replace('.csv', '')] = pd.read_csv(f'{loadingFile}{file}')
    nurse_notes = pd.read_excel('../../fake_notes.xlsx')

    score_dictionary = {}
    scorer = BERTScorer(model_type="bert-base-uncased")
    for i in range(len(nurse_notes)):
        word = ""
        # BERTScore penalizes punctuation, we should take this into account - https://aclanthology.org/2023.findings-acl.381.pdf
        if i < 5:
            candidate =  [re.sub(r'[^\w\s/]', '', i) for i in met_dfs[f'{i % (len(nurse_notes) // 2)}_met']['report']]
            word = "Met"
        else:
            candidate =  [re.sub(r'[^\w\s/]', '', i) for i in unmet_dfs[f'{i % (len(nurse_notes) // 2)}_unmet']['report']]
            word = "Unmet"
        reference = [re.sub(r'[^\w\s/]', '', nurse_notes['Note'][i])] * (len(candidate))
        try:
            P, R, F1 = scorer.score(candidate, reference)
        except Exception as e:
            print(e)
        # save the F1 values, as these are recommended for use by the BERTScore authors - http://arxiv.org/abs/1904.09675
        score_dictionary[f'{i}_{word.lower()}'] = F1.mean()
        print(f"{word} {i} F1 Score: {F1.mean()}")

    # put scores in a pickle file
    with open(savingFile, 'wb') as f:
        pickle.dump(score_dictionary, f)
    

In [5]:
bertScore(gpt4_load, gpt4_save_bert)

Met 0 F1 Score: 0.5171719193458557
Met 1 F1 Score: 0.49404650926589966
Met 2 F1 Score: 0.46354058384895325
Met 3 F1 Score: 0.5251922607421875
Met 4 F1 Score: 0.4995630383491516
Unmet 5 F1 Score: 0.45403069257736206
Unmet 6 F1 Score: 0.5137023329734802
Unmet 7 F1 Score: 0.5214620232582092
Unmet 8 F1 Score: 0.49841681122779846
Unmet 9 F1 Score: 0.48349636793136597


In [6]:
# load scores from pickle file - gpt4
with open(gpt4_save_bert, 'rb') as f:
    loaded_scores = pickle.load(f)
print(loaded_scores)

loaded_scores = pd.DataFrame(loaded_scores, index=['values'])
loaded_scores = loaded_scores.transpose()
print(f"Average BERTScore Met Notes GPT4: {float((((loaded_scores.iloc[:5])['values']).sum()) / (len((loaded_scores.iloc[:5])['values'])))}")
print(f"Average BERTScore Unmet Notes GPT4: {float((((loaded_scores.iloc[5:])['values']).sum()) / (len((loaded_scores.iloc[:5])['values'])))}")
print(f"Average BERTScore All Notes GPT4: {float((((loaded_scores)['values']).sum()) / (len((loaded_scores)['values'])))}")



{'0_met': tensor(0.5172), '1_met': tensor(0.4940), '2_met': tensor(0.4635), '3_met': tensor(0.5252), '4_met': tensor(0.4996), '5_unmet': tensor(0.4540), '6_unmet': tensor(0.5137), '7_unmet': tensor(0.5215), '8_unmet': tensor(0.4984), '9_unmet': tensor(0.4835)}
Average BERTScore Met Notes GPT4: 0.4999028742313385
Average BERTScore Unmet Notes GPT4: 0.49422162771224976
Average BERTScore All Notes GPT4: 0.4970622658729553


In [7]:
bertScore(gpt3_load, gpt3_save_bert)

Met 0 F1 Score: 0.4375615119934082
Met 1 F1 Score: 0.47390180826187134
Met 2 F1 Score: 0.43696340918540955
Met 3 F1 Score: 0.46310850977897644
Met 4 F1 Score: 0.4848701059818268
Unmet 5 F1 Score: 0.47239160537719727
Unmet 6 F1 Score: 0.4976033866405487
Unmet 7 F1 Score: 0.46786484122276306
Unmet 8 F1 Score: 0.4547452926635742
Unmet 9 F1 Score: 0.4616093039512634


In [8]:
# load scores from pickle file - gpt3
with open(gpt3_save_bert, 'rb') as f:
    loaded_scores = pickle.load(f)
print(loaded_scores)
loaded_scores = pd.DataFrame(loaded_scores, index=['values'])
loaded_scores = loaded_scores.transpose()

print(f"Average BERTScore Met Notes GPT3: {float((((loaded_scores.iloc[:5])['values']).sum()) / (len((loaded_scores.iloc[:5])['values'])))}")
print(f"Average BERTScore Unmet Notes GPT3: {float((((loaded_scores.iloc[5:])['values']).sum()) / (len((loaded_scores.iloc[:5])['values'])))}")
print(f"Average BERTScore All Notes GPT3: {float((((loaded_scores)['values']).sum()) / (len((loaded_scores)['values'])))}")


{'0_met': tensor(0.4376), '1_met': tensor(0.4739), '2_met': tensor(0.4370), '3_met': tensor(0.4631), '4_met': tensor(0.4849), '5_unmet': tensor(0.4724), '6_unmet': tensor(0.4976), '7_unmet': tensor(0.4679), '8_unmet': tensor(0.4547), '9_unmet': tensor(0.4616)}
Average BERTScore Met Notes GPT3: 0.4592810571193695
Average BERTScore Unmet Notes GPT3: 0.4708428978919983
Average BERTScore All Notes GPT3: 0.4650619924068451


# Lexical Overlap Metric - METEOR

In [9]:
def calculate_meteor(loadingFile, savingFile):
    # load all files from data folder (generated in Google Colab)
    met_dfs = {}
    unmet_dfs = {}
    for file in os.listdir(loadingFile):
        if '.csv' in file:
            
            if 'unmet' in file:
                unmet_dfs[file.replace('.csv', '')] = pd.read_csv(f'{loadingFile}{file}')
            else:
                met_dfs[file.replace('.csv', '')] = pd.read_csv(f'{loadingFile}{file}')
    nurse_notes = pd.read_excel('../../fake_notes.xlsx')

    score_dictionary = {}
    for i in range(len(nurse_notes)):
        meteor_scores = []
        word = ""
        if i < 5:
            candidates =  [re.sub(r'[^\w\s/]', '', i) for i in met_dfs[f'{i % (len(nurse_notes) // 2)}_met']['report']]
            word = "Met"
        else:
            candidates =  [re.sub(r'[^\w\s/]', '', i) for i in unmet_dfs[f'{i % (len(nurse_notes) // 2)}_unmet']['report']]
            word = "Unmet"
        reference = re.sub(r'[^\w\s/]', '', nurse_notes['Note'][i])
        for candidate in candidates:
            output_score = meteor([word_tokenize(candidate)], word_tokenize(reference))
            meteor_scores.append(output_score)
        try:
            score_dictionary[f'{i % (len(nurse_notes) // 2)}_{word}'] = (sum(meteor_scores)) / (len(meteor_scores))
        except:
            score_dictionary[f'{i % (len(nurse_notes) // 2)}_{word}'] = 0
    print(score_dictionary)

    # put scores in a pickle file
    with open(savingFile, 'wb') as f:
        pickle.dump(score_dictionary, f)

In [10]:
calculate_meteor(gpt3_load, gpt3_save_meteor)

{'0_Met': 0.10735027686834342, '1_Met': 0.1289053087381553, '2_Met': 0.07878988588517691, '3_Met': 0.11689521310927706, '4_Met': 0.1575004960221709, '0_Unmet': 0.06006230884438849, '1_Unmet': 0.1451024352404945, '2_Unmet': 0.09931175308865373, '3_Unmet': 0.10722484110121036, '4_Unmet': 0.1316481612665583}


In [11]:
# load scores from pickle file - gpt3
with open(gpt3_save_meteor, 'rb') as f:
    loaded_scores = pickle.load(f)
print(loaded_scores)
loaded_scores = pd.DataFrame(loaded_scores, index=['values'])
loaded_scores = loaded_scores.transpose()

print(f"Average METEOR Score Met Notes GPT3: {float((((loaded_scores.iloc[:5])['values']).sum()) / (len((loaded_scores.iloc[:5])['values'])))}")
print(f"Average METEOR Score Unmet Notes GPT3: {float((((loaded_scores.iloc[5:])['values']).sum()) / (len((loaded_scores.iloc[:5])['values'])))}")
print(f"Average METEOR Score All Notes GPT3: {float((((loaded_scores)['values']).sum()) / (len((loaded_scores)['values'])))}")

{'0_Met': 0.10735027686834342, '1_Met': 0.1289053087381553, '2_Met': 0.07878988588517691, '3_Met': 0.11689521310927706, '4_Met': 0.1575004960221709, '0_Unmet': 0.06006230884438849, '1_Unmet': 0.1451024352404945, '2_Unmet': 0.09931175308865373, '3_Unmet': 0.10722484110121036, '4_Unmet': 0.1316481612665583}
Average METEOR Score Met Notes GPT3: 0.1178882361246247
Average METEOR Score Unmet Notes GPT3: 0.10866989990826108
Average METEOR Score All Notes GPT3: 0.1132790680164429


In [12]:
calculate_meteor(gpt4_load, gpt4_save_meteor)

{'0_Met': 0.14103607331283205, '1_Met': 0.10709630948768138, '2_Met': 0.07212062398750427, '3_Met': 0.1338196946526854, '4_Met': 0.12877401365322405, '0_Unmet': 0.044324603963744984, '1_Unmet': 0.10920338746430663, '2_Unmet': 0.0961979533528741, '3_Unmet': 0.10095480398334486, '4_Unmet': 0.13141085250575085}


In [13]:
# load scores from pickle file - gpt4
with open(gpt4_save_meteor, 'rb') as f:
    loaded_scores = pickle.load(f)
print(loaded_scores)
loaded_scores = pd.DataFrame(loaded_scores, index=['values'])
loaded_scores = loaded_scores.transpose()

print(f"Average METEOR Score Met Notes GPT4: {float((((loaded_scores.iloc[:5])['values']).sum()) / (len((loaded_scores.iloc[:5])['values'])))}")
print(f"Average METEOR Score Unmet Notes GPT4: {float((((loaded_scores.iloc[5:])['values']).sum()) / (len((loaded_scores.iloc[:5])['values'])))}")
print(f"Average METEOR Score All Notes GPT4: {float((((loaded_scores)['values']).sum()) / (len((loaded_scores)['values'])))}")

{'0_Met': 0.14103607331283205, '1_Met': 0.10709630948768138, '2_Met': 0.07212062398750427, '3_Met': 0.1338196946526854, '4_Met': 0.12877401365322405, '0_Unmet': 0.044324603963744984, '1_Unmet': 0.10920338746430663, '2_Unmet': 0.0961979533528741, '3_Unmet': 0.10095480398334486, '4_Unmet': 0.13141085250575085}
Average METEOR Score Met Notes GPT4: 0.11656934301878544
Average METEOR Score Unmet Notes GPT4: 0.09641832025400429
Average METEOR Score All Notes GPT4: 0.10649383163639485


# Sentiment Analysis - TextBlob

In [14]:
def preprocess_text(text):
    text = re.sub(r'[^\w\s/]', '', text)
    # tokenize
    tokens = word_tokenize(text.lower())
    # remove stop words
    filtered_tokens = [token for token in tokens if token not in stopwords.words('english')]
    # lemmatize the tokens
    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in filtered_tokens]
    # join the tokens back into a string
    processed_text = ' '.join(lemmatized_tokens)
    return processed_text

In [15]:
def sentiment_interpreter(sentiment):
    sentiment = round(sentiment, 2)
    if sentiment > 0.5:
        return "positive"
    elif sentiment < -0.5: 
        return "negative"
    else:
        return "neutral"

In [16]:
def calculate_sentiment(loadingFile, savingFile):
    # load all files from data folder (generated in Google Colab)
    met_dfs = {}
    unmet_dfs = {}
    for file in os.listdir(loadingFile):
        if '.csv' in file:
            
            if 'unmet' in file:
                unmet_dfs[file.replace('.csv', '')] = pd.read_csv(f'{loadingFile}{file}')
            else:
                met_dfs[file.replace('.csv', '')] = pd.read_csv(f'{loadingFile}{file}')
    nurse_notes = pd.read_excel('../../fake_notes.xlsx')

    score_dictionary = {}
    for i in range(len(nurse_notes)):
        nurse_note = preprocess_text(nurse_notes['Note'][i])
        nurse_blob = TextBlob(nurse_note)
        nurse_sentiment = nurse_blob.sentences[0].sentiment.polarity
        nurse_subjectivity = nurse_blob.sentences[0].sentiment.subjectivity
        word = ""
        if i < 5:
            candidates = [(TextBlob(preprocess_text(i))).sentences[0].sentiment.polarity for i in met_dfs[f'{i % (len(nurse_notes) // 2)}_met']['report']]
            try:
                generated_sentiment = (sum(candidates))/ len(candidates)
            except:
                generated_sentiment = 0
            candidates = [(TextBlob(preprocess_text(i))).sentences[0].sentiment.subjectivity for i in met_dfs[f'{i % (len(nurse_notes) // 2)}_met']['report']]
            try:
                generated_subjectivity = (sum(candidates))/ len(candidates)
            except:
                generated_subjectivity = 0
            word = "Met"
        else:
            candidates = [(TextBlob(preprocess_text(i))).sentences[0].sentiment.polarity for i in unmet_dfs[f'{i % (len(nurse_notes) // 2)}_unmet']['report']]
            try:
                generated_sentiment = (sum(candidates))/ len(candidates)
            except:
                generated_sentiment = 0
            candidates = [(TextBlob(preprocess_text(i))).sentences[0].sentiment.subjectivity for i in unmet_dfs[f'{i % (len(nurse_notes) // 2)}_unmet']['report']]
            try:
                generated_subjectivity = (sum(candidates))/ len(candidates)
            except:
                generated_subjectivity = 0
            word = "Unmet"
        score_dictionary[f'{i % (len(nurse_notes) // 2)}_{word}'] = {"Nurse Sentiment": nurse_sentiment, "Generated Sentiment": generated_sentiment, "Absolute Difference in Sentiment": math.sqrt((nurse_sentiment-generated_sentiment) ** 2), "Nurse Subjectivity": nurse_subjectivity , "Generated Subjectivity": generated_subjectivity, "Absolute Difference in Subjectivity":math.sqrt((nurse_subjectivity-generated_subjectivity) ** 2)}
    print(score_dictionary)
    # put scores in a pickle file
    with open(savingFile, 'wb') as f:
        pickle.dump(score_dictionary, f)

In [17]:
calculate_sentiment(gpt3_load, gpt3_save_sentiment)

{'0_Met': {'Nurse Sentiment': 0.023333333333333317, 'Generated Sentiment': 0.12996527777777778, 'Absolute Difference in Sentiment': 0.10663194444444446, 'Nurse Subjectivity': 0.4866666666666667, 'Generated Subjectivity': 0.41060630341880344, 'Absolute Difference in Subjectivity': 0.07606036324786325}, '1_Met': {'Nurse Sentiment': 0.43333333333333335, 'Generated Sentiment': 0.15375, 'Absolute Difference in Sentiment': 0.27958333333333335, 'Nurse Subjectivity': 0.5416666666666666, 'Generated Subjectivity': 0.4215064102564103, 'Absolute Difference in Subjectivity': 0.12016025641025635}, '2_Met': {'Nurse Sentiment': 0.24444444444444444, 'Generated Sentiment': 0.10806944444444444, 'Absolute Difference in Sentiment': 0.136375, 'Nurse Subjectivity': 0.2638888888888889, 'Generated Subjectivity': 0.3267083333333333, 'Absolute Difference in Subjectivity': 0.06281944444444443}, '3_Met': {'Nurse Sentiment': 0.08484848484848484, 'Generated Sentiment': 0.15111111111111114, 'Absolute Difference in Se

In [18]:
def sentiment_parser(loaded_scores):
    nurse_met_sentiment = 0
    generated_met_sentiment = 0
    nurse_met_subjectivity = 0
    generated_met_subjectivity = 0
    met_count = 0
    nurse_unmet_sentiment = 0
    generated_unmet_sentiment = 0
    nurse_unmet_subjectivity = 0
    generated_unmet_subjectivity = 0
    unmet_count = 0
    for score in loaded_scores:
        if 'Unmet' in score:
            nurse_unmet_sentiment += loaded_scores[score]['Nurse Sentiment']
            generated_unmet_sentiment += loaded_scores[score]['Generated Sentiment']
            nurse_unmet_subjectivity += loaded_scores[score]['Nurse Subjectivity']
            generated_unmet_subjectivity += loaded_scores[score]['Generated Subjectivity']
            unmet_count += 1

        elif 'Met' in score:
            # print(loaded_scores[score]['Generated Sentiment'])
            nurse_met_sentiment += loaded_scores[score]['Nurse Sentiment']
            generated_met_sentiment += loaded_scores[score]['Generated Sentiment']
            nurse_met_subjectivity += loaded_scores[score]['Nurse Subjectivity']
            generated_met_subjectivity += loaded_scores[score]['Generated Subjectivity']
            met_count += 1
    # met note scores
    print("\n------------\nMet Notes\n------------\n")
    print(f"Nurse Met Sentiment: {nurse_met_sentiment / met_count}")
    print(f"Generated Met Sentiment: {generated_met_sentiment / met_count}")
    print(f"Difference in Met Sentiment: {math.sqrt(((nurse_met_sentiment / met_count)-(generated_met_sentiment / met_count)) ** 2)}")
    print(f"Nurse Met Subjectivity: {nurse_met_subjectivity / met_count}")
    print(f"Generated Met Subjectivity: {generated_met_subjectivity / met_count}")
    print(f"Difference in Met Subjectivity: {math.sqrt(((nurse_met_subjectivity / met_count)-(generated_met_subjectivity / met_count)) ** 2)}")

    # unmet note scores
    print("\n------------\nUnmet Notes\n------------\n")
    print(f"Nurse Unmet Sentiment: {nurse_unmet_sentiment / unmet_count}")
    print(f"Generated Unmet Sentiment: {generated_unmet_sentiment / unmet_count}")
    print(f"Difference in Unmet Sentiment: {math.sqrt(((nurse_unmet_sentiment / unmet_count)-(generated_unmet_sentiment / unmet_count)) ** 2)}")
    print(f"Nurse Unmet Subjectivity: {nurse_unmet_subjectivity / unmet_count}")
    print(f"Generated Unmet Subjectivity: {generated_unmet_subjectivity / unmet_count}")
    print(f"Difference in Unmet Subjectivity: {math.sqrt(((nurse_unmet_subjectivity / unmet_count)-(generated_unmet_subjectivity / unmet_count)) ** 2)}")


In [19]:
# load scores from pickle file - gpt4
with open(gpt3_save_sentiment, 'rb') as f:
    loaded_scores = pickle.load(f)
print("GPT3")
print(loaded_scores)
sentiment_parser(loaded_scores=loaded_scores)

GPT3
{'0_Met': {'Nurse Sentiment': 0.023333333333333317, 'Generated Sentiment': 0.12996527777777778, 'Absolute Difference in Sentiment': 0.10663194444444446, 'Nurse Subjectivity': 0.4866666666666667, 'Generated Subjectivity': 0.41060630341880344, 'Absolute Difference in Subjectivity': 0.07606036324786325}, '1_Met': {'Nurse Sentiment': 0.43333333333333335, 'Generated Sentiment': 0.15375, 'Absolute Difference in Sentiment': 0.27958333333333335, 'Nurse Subjectivity': 0.5416666666666666, 'Generated Subjectivity': 0.4215064102564103, 'Absolute Difference in Subjectivity': 0.12016025641025635}, '2_Met': {'Nurse Sentiment': 0.24444444444444444, 'Generated Sentiment': 0.10806944444444444, 'Absolute Difference in Sentiment': 0.136375, 'Nurse Subjectivity': 0.2638888888888889, 'Generated Subjectivity': 0.3267083333333333, 'Absolute Difference in Subjectivity': 0.06281944444444443}, '3_Met': {'Nurse Sentiment': 0.08484848484848484, 'Generated Sentiment': 0.15111111111111114, 'Absolute Difference 

In [20]:
calculate_sentiment(gpt4_load, gpt4_save_sentiment)

{'0_Met': {'Nurse Sentiment': 0.023333333333333317, 'Generated Sentiment': 0.06554478286264, 'Absolute Difference in Sentiment': 0.04221144952930668, 'Nurse Subjectivity': 0.4866666666666667, 'Generated Subjectivity': 0.4088945686588544, 'Absolute Difference in Subjectivity': 0.07777209800781232}, '1_Met': {'Nurse Sentiment': 0.43333333333333335, 'Generated Sentiment': 0.07989401669758812, 'Absolute Difference in Sentiment': 0.35343931663574524, 'Nurse Subjectivity': 0.5416666666666666, 'Generated Subjectivity': 0.39662928539714254, 'Absolute Difference in Subjectivity': 0.1450373812695241}, '2_Met': {'Nurse Sentiment': 0.24444444444444444, 'Generated Sentiment': 0.10115513124441695, 'Absolute Difference in Sentiment': 0.14328931320002747, 'Nurse Subjectivity': 0.2638888888888889, 'Generated Subjectivity': 0.4232001956203084, 'Absolute Difference in Subjectivity': 0.15931130673141952}, '3_Met': {'Nurse Sentiment': 0.08484848484848484, 'Generated Sentiment': 0.031601870285798854, 'Absol

In [21]:
# load scores from pickle file - gpt4
with open(gpt4_save_sentiment, 'rb') as f:
    loaded_scores = pickle.load(f)
print("GPT4")
print(loaded_scores)
sentiment_parser(loaded_scores=loaded_scores)

GPT4
{'0_Met': {'Nurse Sentiment': 0.023333333333333317, 'Generated Sentiment': 0.06554478286264, 'Absolute Difference in Sentiment': 0.04221144952930668, 'Nurse Subjectivity': 0.4866666666666667, 'Generated Subjectivity': 0.4088945686588544, 'Absolute Difference in Subjectivity': 0.07777209800781232}, '1_Met': {'Nurse Sentiment': 0.43333333333333335, 'Generated Sentiment': 0.07989401669758812, 'Absolute Difference in Sentiment': 0.35343931663574524, 'Nurse Subjectivity': 0.5416666666666666, 'Generated Subjectivity': 0.39662928539714254, 'Absolute Difference in Subjectivity': 0.1450373812695241}, '2_Met': {'Nurse Sentiment': 0.24444444444444444, 'Generated Sentiment': 0.10115513124441695, 'Absolute Difference in Sentiment': 0.14328931320002747, 'Nurse Subjectivity': 0.2638888888888889, 'Generated Subjectivity': 0.4232001956203084, 'Absolute Difference in Subjectivity': 0.15931130673141952}, '3_Met': {'Nurse Sentiment': 0.08484848484848484, 'Generated Sentiment': 0.031601870285798854, '